# Data Governance Challenge: enriquecimiento de descripciones de productos

**Flujo:** MercadoLibre *Buscador de productos* (`/products/search`, catálogo curado por MELI), selección de ítems a enriquecer, Gemini (con retry/backoff), SQLite (persistencia) y export a JSON (fuente para la futura API RESTful).

**Autenticación:** este endpoint requiere OAuth2 (`Authorization: Bearer`). MercadoLibre solo soporta el grant type `authorization_code` (no `client_credentials`), así que hace falta un login único por navegador para obtener un `refresh_token`. A partir de ahí, el `access_token` se renueva solo en cada corrida sin volver a loguearte (ver sección 3).

**Observabilidad:** logging estructurado y métricas de la corrida (extraídos, enriquecidos, omitidos, errores). En producción esto se llevaría a un colector (CloudWatch/Datadog) y las métricas a Prometheus/StatsD.


## 1. Instalación de dependencias

In [ ]:
!pip install -q requests google-genai

## 2. Imports y configuración

En Colab, cargá `GEMINI_API_KEY`, `MELI_CLIENT_ID` y `MELI_CLIENT_SECRET` en **secrets** (ícono de llave en el panel izquierdo) antes de correr esta celda. `MELI_REFRESH_TOKEN` se lee más adelante, en la sección 3, porque cambia en cada corrida.

In [ ]:
import json
import logging
import os
import sqlite3
import time
from contextlib import closing
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Final, Optional

import requests
from google import genai

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s")
logger = logging.getLogger("meli_enrichment")


def _secret(name: str) -> str:
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except ImportError:
        return os.environ.get(name, "").strip()


GEMINI_API_KEY = _secret("GEMINI_API_KEY")
MELI_CLIENT_ID = _secret("MELI_CLIENT_ID")
MELI_CLIENT_SECRET = _secret("MELI_CLIENT_SECRET")

if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY no configurada (Colab secrets o variable de entorno)")
if not MELI_CLIENT_ID or not MELI_CLIENT_SECRET:
    raise RuntimeError("MELI_CLIENT_ID / MELI_CLIENT_SECRET no configuradas (Colab secrets o variable de entorno)")

# Parámetros de la corrida. Se centralizan acá para no tener valores sueltos en el código.
MELI_SITE_ID: Final[str] = "MLA"              # Argentina. Cambiar según el país objetivo.
MELI_OAUTH_TOKEN_URL: Final[str] = "https://api.mercadolibre.com/oauth/token"
MELI_AUTHORIZE_URL: Final[str] = "https://auth.mercadolibre.com.ar/authorization"
MELI_REDIRECT_URI: Final[str] = "https://www.example.com/"  # Debe coincidir con el registrado en tu app.

SEARCH_QUERY: Final[str] = "notebook gamer"   # Categoría/búsqueda a extraer.
MAX_ITEMS: Final[int] = 50                    # Tope de ítems a traer en esta corrida.
MELI_PAGE_SIZE: Final[int] = 50               # Máximo permitido por la API de búsqueda.

DB_PATH: Final[str] = "meli_products.db"
EXPORT_JSON_PATH: Final[str] = "enriched_products_export.json"

MIN_DESCRIPTION_LENGTH: Final[int] = 60       # Umbral para decidir si "vale la pena" enriquecer.
MAX_DESCRIPTION_LENGTH: Final[int] = 400      # Límite duro de la descripción enriquecida (se valida, no solo se pide).
# gemini-flash-lite-latest: variante más económica y rápida de la familia Gemini Flash.
# Alcanza para este caso de uso (descripciones cortas, con restricciones estrictas de
# formato) y evita agotar cuota de tokens durante las pruebas del challenge.
GEMINI_MODEL_NAME: Final[str] = "gemini-flash-lite-latest"
MAX_RETRIES: Final[int] = 3

# google-generativeai fue deprecado por Google en favor de este SDK unificado (google-genai).
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

## 3. Autenticación con MercadoLibre (OAuth)

`/products/search` exige un `access_token` válido. MercadoLibre solo soporta los grant types
`authorization_code` y `refresh_token` (no existe `client_credentials` para server-to-server).
Esto implica un login por navegador **una única vez** para obtener un `refresh_token`, que después
se usa indefinidamente para generar `access_token` nuevos (duran ~6hs) sin volver a loguearte.

**Pre-requisito:** tener una app creada en [developers.mercadolibre.com](https://developers.mercadolibre.com/),
con `MELI_REDIRECT_URI` (definida arriba) configurada como Redirect URI de la app.

### 3a. Primera vez: obtener el `refresh_token` (correr una sola vez)

Ejecutá esta celda: te va a imprimir una URL, la abrís, iniciás sesión con tu cuenta de MELI,
autorizás la app, y pegás la URL completa a la que te redirige (aunque la página de destino
muestre un error, el `code` va a estar igual en la barra de direcciones).

In [ ]:
_auth_url = (
    f"{MELI_AUTHORIZE_URL}?response_type=code&client_id={MELI_CLIENT_ID}"
    f"&redirect_uri={MELI_REDIRECT_URI}"
)
print("Abrí esta URL, autorizá la app, y volvé acá:")
print(_auth_url)

from urllib.parse import urlparse, parse_qs

_redirect_url = input("Pegá la URL completa de redirección: ").strip()
_query_params = parse_qs(urlparse(_redirect_url).query)
if "code" not in _query_params:
    raise ValueError("No se encontró el parámetro 'code' en la URL pegada. Revisá que sea la URL completa.")

_token_response = requests.post(
    MELI_OAUTH_TOKEN_URL,
    data={
        "grant_type": "authorization_code",
        "client_id": MELI_CLIENT_ID,
        "client_secret": MELI_CLIENT_SECRET,
        "code": _query_params["code"][0],
        "redirect_uri": MELI_REDIRECT_URI,
    },
    timeout=15,
)
_token_response.raise_for_status()

print("Listo. Guardá este valor como el secret 'MELI_REFRESH_TOKEN' en Colab y no vuelvas a correr esta celda:")
print(_token_response.json()["refresh_token"])

### 3b. Cada corrida: renovar el `access_token` a partir del `refresh_token`

**El `refresh_token` de MELI es de un solo uso**: cada vez que lo usás, el servidor te devuelve
uno **nuevo** y el anterior queda inválido. Esta celda lee el secret al momento de correrla (no
hace falta volver a la celda 2), y al final te imprime el `refresh_token` nuevo. Actualizá el
secret `MELI_REFRESH_TOKEN` con ese valor antes de la próxima vez que corras el notebook, o la
siguiente renovación va a fallar con `invalid_grant`.

In [ ]:
def refresh_meli_access_token(client_id: str, client_secret: str, refresh_token: str) -> tuple[str, str]:
    """Devuelve (access_token, refresh_token_nuevo). El refresh_token es de un solo uso: MELI
    invalida el que se envía y entrega uno nuevo en cada llamada, que hay que persistir."""
    response = requests.post(
        MELI_OAUTH_TOKEN_URL,
        data={
            "grant_type": "refresh_token",
            "client_id": client_id,
            "client_secret": client_secret,
            "refresh_token": refresh_token,
        },
        timeout=15,
    )
    if not response.ok:
        raise RuntimeError(
            "No se pudo renovar el access_token de MELI "
            f"(status {response.status_code}): {response.text}. "
            "El refresh_token es de un solo uso: si ya se usó antes, repetí el paso 3a."
        )
    tokens = response.json()
    return tokens["access_token"], tokens["refresh_token"]

MELI_REFRESH_TOKEN = _secret("MELI_REFRESH_TOKEN")  # Se lee acá, no en la celda 2, porque rota.
if not MELI_REFRESH_TOKEN:
    raise RuntimeError("MELI_REFRESH_TOKEN no configurado. Corré la sección 3a una vez y guardalo en secrets.")

MELI_ACCESS_TOKEN, _new_meli_refresh_token = refresh_meli_access_token(
    MELI_CLIENT_ID, MELI_CLIENT_SECRET, MELI_REFRESH_TOKEN
)
logger.info("access_token de MELI renovado correctamente")
print("Actualizá el secret 'MELI_REFRESH_TOKEN' con este valor antes de la próxima corrida:")
print(_new_meli_refresh_token)

## 4. Modelo de datos

In [ ]:
class ProductStatus(str, Enum):
    PENDING = "pending"
    ENRICHED = "enriched"
    SKIPPED = "skipped"
    ERROR = "error"


@dataclass
class Product:
    item_id: str
    name: str
    price: Optional[float]
    currency: Optional[str]
    image_url: Optional[str]
    permalink: Optional[str]
    rating: Optional[float] = None  # No expuesto por /products/{id}; queda en None (ver informe).
    original_description: str = ""
    specifications: dict[str, str] = field(default_factory=dict)
    enriched_description: Optional[str] = None
    status: ProductStatus = ProductStatus.PENDING
    error_message: Optional[str] = None

## 5. Extracción de datos: Buscador de productos de MercadoLibre

Usa `/products/search` (catálogo curado por MELI, con ficha técnica) para descubrir productos y
`/products/{id}` para el detalle completo (specs, imágenes y `short_description`).
Incluye backoff exponencial ante `429` (rate limit) y `5xx`. Todas las llamadas van autenticadas
con el `access_token` obtenido en la sección 3.

In [ ]:
def _exponential_backoff_seconds(attempt: int) -> int:
    return 2 ** attempt


class MeliClient:
    """Cliente delgado sobre el Buscador de productos de MercadoLibre (requiere OAuth)."""

    BASE_URL: Final[str] = "https://api.mercadolibre.com"

    def __init__(self, access_token: str, session: Optional[requests.Session] = None):
        self.session = session or requests.Session()
        self.session.headers["Authorization"] = f"Bearer {access_token}"

    def _get(self, url: str, params: Optional[dict] = None, max_retries: int = MAX_RETRIES) -> dict:
        """GET con reintentos ante rate limiting (429) y errores transitorios (5xx)."""
        last_status: Optional[int] = None

        for attempt in range(1, max_retries + 1):
            response = self.session.get(url, params=params, timeout=15)
            last_status = response.status_code

            if response.ok:
                return response.json()

            if response.status_code == 429 or response.status_code >= 500:
                wait_seconds = _exponential_backoff_seconds(attempt)
                logger.warning(
                    "GET %s devolvió %s (intento %d/%d), reintentando en %ds",
                    url, response.status_code, attempt, max_retries, wait_seconds,
                )
                time.sleep(wait_seconds)
                continue

            response.raise_for_status()  # Errores 4xx no recuperables (ej. 401 token vencido, 404).

        raise RuntimeError(f"GET {url} falló tras {max_retries} intentos (último status: {last_status})")

    def search_product_ids(self, query: str, site_id: str, max_items: int) -> list[str]:
        """Pagina /products/search y devuelve los product_id encontrados."""
        product_ids: list[str] = []
        offset = 0

        while len(product_ids) < max_items:
            data = self._get(
                f"{self.BASE_URL}/products/search",
                params={
                    "status": "active",  # Solo productos comprables/publicables, no descontinuados.
                    "site_id": site_id,
                    "q": query,
                    "limit": min(MELI_PAGE_SIZE, max_items - len(product_ids)),
                    "offset": offset,
                },
            )
            results = data.get("results", [])
            if not results:
                break

            product_ids.extend(result["id"] for result in results)
            offset += len(results)
            if offset >= data.get("paging", {}).get("total", 0):
                break

        return product_ids[:max_items]

    def get_product_detail(self, product_id: str) -> dict:
        return self._get(f"{self.BASE_URL}/products/{product_id}")

    def build_product(self, product_id: str) -> Product:
        detail = self.get_product_detail(product_id)

        specifications = {
            attribute["name"]: attribute["value_name"]
            for attribute in detail.get("attributes", [])
            if attribute.get("name") and attribute.get("value_name")
        }

        pictures = detail.get("pictures", [])
        buy_box_winner = detail.get("buy_box_winner") or {}  # Puede no haber vendedor ganador.

        return Product(
            item_id=detail["id"],
            name=detail.get("name", ""),
            price=buy_box_winner.get("price"),
            currency=buy_box_winner.get("currency_id"),
            image_url=pictures[0]["url"] if pictures else None,
            permalink=detail.get("permalink"),
            original_description=(detail.get("short_description") or {}).get("content", "") or "",
            specifications=specifications,
        )

## 6. Selección: ¿qué ítems enriquecer?

Regla simple y auditable: se enriquecen los ítems con descripción ausente o corta.
Esto es clave para el pilar de **eficiencia operativa**: no se gasta cuota de Gemini en ítems que ya están bien descriptos.

In [ ]:
def needs_enrichment(product: Product) -> bool:
    return len(product.original_description.strip()) < MIN_DESCRIPTION_LENGTH

## 7. Enriquecimiento: API de Gemini

Prompt en inglés, con restricciones explícitas de tono, longitud y "no inventar atributos"
(esto conecta con la nota del challenge sobre ética y precisión en la generación).

In [ ]:
class DescriptionEnricher:
    def __init__(self, client: genai.Client, model_name: str = GEMINI_MODEL_NAME):
        self._client = client
        self._model_name = model_name

    @staticmethod
    def _build_prompt(product: Product) -> str:
        specs = "; ".join(f"{k}: {v}" for k, v in product.specifications.items()) or "N/A"
        return (
            "You are an e-commerce copywriter. Generate an enriched product description "
            "for clarity and engagement, to be consumed by a recommendation system.\n\n"
            f"Product name: {product.name}\n"
            f"Price: {product.price} {product.currency}\n"
            f"Specifications: {specs}\n"
            f"Original description (may be empty or low quality): {product.original_description}\n\n"
            "Requirements:\n"
            "- Tone: neutral-professional, aimed at online shoppers comparing similar items.\n"
            "- Valid sources of information are the product name, the specifications, and the "
            "original description -- do NOT invent features absent from all three. If the product "
            "name conflicts with a specification (e.g. a specification value that is clearly "
            "implausible, like a gaming laptop with a 1 GB hard drive), prefer the product name: "
            "specifications may contain data entry errors from the source catalog.\n"
            "- Length: 2-3 complete sentences, aiming for 400 characters or fewer. A small overage "
            "is acceptable if needed to finish the last sentence naturally -- never cut a sentence "
            "short just to fit the limit.\n"
            "- Output plain text only, no markdown, no bullet points.\n"
            "- Vary the opening of the description: do not default to the same opening word or "
            "marketing phrase (e.g. do not always start with 'Notebook', 'Eleva', 'Potencia', or "
            "similar) across different products -- write a distinct opening for this specific item.\n"
            "- Output language: Spanish (the marketplace is MercadoLibre Argentina, site MLA), "
            "regardless of the fact these instructions are written in English.\n"
        )

    def generate(self, product: Product, max_retries: int = MAX_RETRIES) -> str:
        prompt = self._build_prompt(product)
        last_text: Optional[str] = None

        for attempt in range(1, max_retries + 1):
            try:
                response = self._client.models.generate_content(model=self._model_name, contents=prompt)
                text = (response.text or "").strip()
                if not text:
                    raise ValueError("Respuesta vacía de Gemini")
            except Exception as exc:  # El SDK de Gemini expone excepciones heterogéneas.
                wait_seconds = _exponential_backoff_seconds(attempt)
                logger.warning(
                    "[%s] Error generando descripción (intento %d/%d): %s. Reintentando en %ds",
                    product.item_id, attempt, max_retries, exc, wait_seconds,
                )
                time.sleep(wait_seconds)
                continue

            if len(text) <= MAX_DESCRIPTION_LENGTH:
                return text

            last_text = text
            logger.warning(
                "[%s] Descripción de %d caracteres excede el límite de %d (intento %d/%d), reintentando",
                product.item_id, len(text), MAX_DESCRIPTION_LENGTH, attempt, max_retries,
            )

        if last_text:
            # Preferimos una descripción algo más larga que lo pedido a una cortada a mitad de
            # una idea: truncar arruinaría la calidad de lectura, que es peor que un exceso menor.
            logger.warning(
                "[%s] Ninguna generación respetó el límite de %d caracteres tras %d intentos; "
                "se usa la última generada sin truncar (%d caracteres).",
                product.item_id, MAX_DESCRIPTION_LENGTH, max_retries, len(last_text),
            )
            return last_text

        raise RuntimeError(f"No se pudo generar descripción para {product.item_id} tras {max_retries} intentos")

## 8. Persistencia: SQLite

Tabla única `products` con `status` y `error_message` por fila: una corrida fallida queda
auditable sin ir a buscar en los logs. `upsert` con `ON CONFLICT` la hace idempotente.
Cada operación de escritura corre dentro de una transacción (`with self._connection:`), que
en `sqlite3` hace commit automático al salir del bloque o rollback si hubo una excepción.

In [ ]:
class ProductStore:
    def __init__(self, db_path: str = DB_PATH):
        self._connection = sqlite3.connect(db_path)
        with self._connection:
            self._connection.execute("""
                CREATE TABLE IF NOT EXISTS products (
                    item_id TEXT PRIMARY KEY,
                    name TEXT,
                    price REAL,
                    currency TEXT,
                    image_url TEXT,
                    permalink TEXT,
                    rating REAL,
                    original_description TEXT,
                    specifications_json TEXT,
                    enriched_description TEXT,
                    status TEXT,
                    error_message TEXT,
                    created_at TEXT,
                    updated_at TEXT
                )
            """)

    def close(self) -> None:
        self._connection.close()

    def upsert(self, product: Product) -> None:
        now = datetime.now(timezone.utc).isoformat()
        with self._connection:
            self._connection.execute(
                """
                INSERT INTO products (
                    item_id, name, price, currency, image_url, permalink, rating,
                    original_description, specifications_json, enriched_description,
                    status, error_message, created_at, updated_at
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                ON CONFLICT(item_id) DO UPDATE SET
                    name=excluded.name,
                    price=excluded.price,
                    currency=excluded.currency,
                    image_url=excluded.image_url,
                    permalink=excluded.permalink,
                    rating=excluded.rating,
                    original_description=excluded.original_description,
                    specifications_json=excluded.specifications_json,
                    enriched_description=excluded.enriched_description,
                    status=excluded.status,
                    error_message=excluded.error_message,
                    updated_at=excluded.updated_at
                """,
                (
                    product.item_id, product.name, product.price, product.currency,
                    product.image_url, product.permalink, product.rating,
                    product.original_description,
                    json.dumps(product.specifications, ensure_ascii=False),
                    product.enriched_description, product.status.value, product.error_message,
                    now, now,
                ),
            )

    def export_json(self, path: str = EXPORT_JSON_PATH) -> None:
        """Snapshot desnormalizado, pensado como fuente de lectura para la futura API."""
        rows = self._connection.execute(
            """
            SELECT item_id, name, price, currency, image_url, permalink, rating,
                   enriched_description, original_description, specifications_json, status
            FROM products
            """
        ).fetchall()

        products = [
            {
                "item_id": item_id,
                "name": name,
                "price": price,
                "currency": currency,
                "image_url": image_url,
                "permalink": permalink,
                "rating": rating,
                "description": enriched_description or original_description,
                "specifications": json.loads(specifications_json) if specifications_json else {},
                "status": status,
            }
            for (
                item_id, name, price, currency, image_url, permalink, rating,
                enriched_description, original_description, specifications_json, status,
            ) in rows
        ]

        with open(path, "w", encoding="utf-8") as f:
            json.dump(products, f, ensure_ascii=False, indent=2)
        logger.info("Export generado: %s (%d ítems)", path, len(products))

## 9. Orquestación principal

Junta extracción → selección → enriquecimiento → persistencia, y lleva las métricas
de la corrida (`extracted`, `enriched`, `skipped`, `errors`).

In [ ]:
def run_pipeline() -> dict[str, int]:
    metrics = {"extracted": 0, "enriched": 0, "skipped": 0, "errors": 0}

    meli_client = MeliClient(access_token=MELI_ACCESS_TOKEN)
    enricher = DescriptionEnricher(gemini_client)

    with closing(ProductStore()) as store:
        logger.info("Buscando productos para query=%r en site=%s", SEARCH_QUERY, MELI_SITE_ID)
        product_ids = meli_client.search_product_ids(SEARCH_QUERY, MELI_SITE_ID, MAX_ITEMS)
        logger.info("%d productos encontrados", len(product_ids))

        for product_id in product_ids:
            try:
                product = meli_client.build_product(product_id)
                metrics["extracted"] += 1
            except Exception as exc:
                logger.error("[%s] Error extrayendo detalle: %s", product_id, exc)
                metrics["errors"] += 1
                continue

            if not needs_enrichment(product):
                product.status = ProductStatus.SKIPPED
                metrics["skipped"] += 1
                store.upsert(product)
                continue

            try:
                product.enriched_description = enricher.generate(product)
                product.status = ProductStatus.ENRICHED
                metrics["enriched"] += 1
            except Exception as exc:
                product.status = ProductStatus.ERROR
                product.error_message = str(exc)
                metrics["errors"] += 1
                logger.error("[%s] %s", product_id, exc)

            store.upsert(product)

        store.export_json()

    logger.info("Corrida finalizada. Métricas: %s", metrics)
    return metrics

## 10. Ejecutar el pipeline

In [ ]:
metrics = run_pipeline()
metrics

## 11. Verificación rápida de resultados

In [ ]:
import pandas as pd

with closing(sqlite3.connect(DB_PATH)) as conn:
    df = pd.read_sql_query(
        "SELECT item_id, name, status, original_description, enriched_description FROM products",
        conn,
    )

df.head(10)